# Semaine 7 — Jour 4 : Fiabilité, reprises et erreurs

Objectif : Séparer erreurs transitoires et erreurs permanentes.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Pour une étude complète, prévoir 2 à 3 heures avec entretien, challenge et revue. Le premier lancement Docker peut demander un temps supplémentaire.

Prérequis : Python 3.12 recommandé, HTTP/JSON, notions d’agents et de persistance des semaines précédentes. Les commandes Python utilisent uniquement la bibliothèque standard. n8n exige Docker Compose v2 pour le lab d’intégration.

Lire [les objectifs](learning_objectives.md), [le cours](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md) avant d’ouvrir [les corrigés](corriges/exercises_solution.md). Préparer [l’entretien](interview.md) et [le challenge](challenge.md).

Notebooks depuis la racine : `notebooks/week07/S7_J4_fiabilite_reprises.ipynb` et `notebooks/week07/teacher/S7_J4_fiabilite_reprises_teacher.ipynb`.

Critère de réussite : Décrire l’outbox et la fenêtre de panne d’un effet externe.

Le lab ne requiert aucune clé LLM. Ses réponses sont simulées et ne mesurent pas la qualité d’un modèle réel.


# Objectifs mesurables

- Séparer erreurs transitoires et erreurs permanentes.
- Mettre en œuvre une reprise bornée et une quarantaine observable.
- Décrire l’outbox et la fenêtre de panne d’un effet externe.

## Preuves attendues

Une sortie de lab reproductible, une explication de l’invariant étudié et une limite correctement identifiée. Le challenge approfondit le parcours essentiel sans être nécessaire pour finir la session de 60 minutes.


# Semaine 7 — Jour 4 : Fiabilité, reprises et erreurs


## 1. Toutes les erreurs ne se rejouent pas
Un timeout ou une indisponibilité temporaire peut disparaître. Un contrat invalide ne se corrige pas avec une attente. Le moteur traite TemporaryFailure comme une erreur temporaire et InvalidEvent comme un défaut permanent. Un adaptateur fournisseur doit convertir les erreurs pertinentes dans ces types; il ne doit pas transformer toutes les exceptions en « retry ».

Les erreurs de programmation inattendues remontent à l’appelant. Les cacher dans une boucle de reprise rendrait le diagnostic difficile et pourrait répéter des actions incorrectes. Sur n8n, une erreur de transport vers le backend relève du workflow technique; sur le backend, une erreur du générateur relève du job métier. Ces deux niveaux doivent être observés séparément.

## 2. Reprise bornée et dates persistées
Le moteur augmente attempts puis inscrit ready_at. Après les échecs successifs, les délais sont de 2 puis 4 secondes; le troisième échec termine en dead_letter. Trois tentatives signifient un appel initial et deux reprises, pas trois reprises supplémentaires. Le scheduler ne doit pas traiter un job avant ready_at.

Le lab n’effectue aucun sleep dans le traitement. Il libère la main et laisse le prochain tick trouver le job éligible. Les tests utilisent une horloge injectable : augmenter une valeur reproduit un délai sans ralentir la suite. En production, ajouter du jitter et respecter les consignes du fournisseur, notamment Retry-After si disponible. Sans jitter, une panne commune peut provoquer une reprise simultanée de tous les clients.

## 3. Les reprises imbriquées multiplient les appels
Si un SDK retente trois fois et que le workflow retente trois fois l’appel entier, le fournisseur peut recevoir jusqu’à neuf appels. Il faut définir le propriétaire du budget de reprise et une durée totale. Dans ce lab, le moteur possède le budget métier. Un échec du POST /tick peut laisser des jobs déjà traités : le tick suivant vérifie leur état au lieu de générer systématiquement à nouveau.

Le traitement est limité à cent jobs éligibles par tick. C’est une borne de batch, pas un mécanisme complet de backpressure. Il n’y a ni quotas par client, ni ordonnancement équitable, ni workers parallèles. Si le débit d’entrée dépasse celui du traitement, la file durable grossit. Le compteur des états permet de voir l’accumulation, mais une exploitation réelle ajouterait l’âge du plus ancien job.

## 4. Quarantaine et replay
Dead_letter signifie que le traitement automatique est arrêté. Un opérateur examine le contrat, les tentatives et le type d’erreur. Après correction de la cause, il demande un replay explicite. Le replay remet le job à received et remet le compteur à zéro; cette action apparaît dans l’audit. Le endpoint est protégé par le token de relecture du lab.

La séparation entre une dead letter et un refus humain est importante. Le refus est un résultat métier accepté; il ne doit pas devenir un incident à retenter. Une validation expirée ne se relance pas automatiquement avec une ancienne autorisation.

## 5. Outbox et effets externes
Le lab écrit la publication simulée dans la même base que le job. Avec un destinataire externe, la transaction locale ne pourrait pas inclure son acceptation réseau. Une outbox enregistre une intention de publication avec la transition métier, puis un dispatcher tente l’envoi. Cette intention durable supprime la perte entre décision locale et lancement de l’envoi, mais le dispatcher peut toujours répéter un envoi. Le destinataire doit dédupliquer ou le système doit réconcilier.

L’outbox n’est pas implémentée dans ce moteur pédagogique; elle est étudiée comme une extension. Le cours distingue volontairement l’invariant démontré localement de la garantie attendue pour un service tiers.

## À retenir
Une reprise fiable possède une catégorie d’erreur, un budget, une date d’éligibilité et une sortie observable. Un replay est une opération contrôlée, pas une boucle infinie.


## Diagramme du jour

```mermaid
flowchart TD
  A["Traitement"] --> B{"Résultat"}
  B -->|valide| C["Validation humaine"]
  B -->|temporaire| D{"Budget restant ?"}
  D -->|oui| E["Date de reprise"]
  E --> A
  D -->|non| F["Dead letter"]
  B -->|contrat invalide| F
```

Si votre lecteur Jupyter ne rend pas Mermaid, ouvrir le fichier .mmd avec un lecteur compatible.

# Exemple exécutable autonome

Ce fragment illustre le concept du jour. Le moteur complet et ses scénarios sont dans examples/week07.



In [ ]:
now = 1000
attempts = 0
ready_at = now
history = []
while attempts < 3:
    now = max(now, ready_at)
    attempts += 1
    history.append(now)
    ready_at = now + 2 ** attempts
assert history == [1000, 1002, 1006]
print({"attempt_times": history, "terminal": "dead_letter"})


# Lab J4 — Fiabilité, reprises et erreurs

Depuis la racine du dépôt :

```bash
python3 book/week07/day04/labs/run.py
```

Résultat attendu : **calls=2, status=awaiting_review**.

Le script exécute le scénario du jour contre une base en mémoire et ferme la connexion. Il ne crée aucun message externe. Inspecter `examples/week07/demo.py` et la méthode correspondante dans `workflow.py` pour relier la sortie à la transition étudiée.

## Travail à réaliser

1. Exécuter le scénario et expliquer chaque champ retourné.
2. Réaliser l’exercice 2 du jour avec une entrée différente.
3. Justifier l’absence de publication non autorisée.

Pour le parcours n8n, suivre [le guide d’installation](../../../../docs/week07/QUICKSTART.md). J1 configure la réception; J2 ajoute le scheduler; J3 teste les doublons; J4 observe les reprises et erreurs; J5 étudie le port de génération; J6 importe le workflow de décision; J7 exécute la checklist d’intégration.

Le scénario J4 utilise une horloge virtuelle. Les tests de reprise sont rapides et n’attendent pas réellement les délais. Le projet HTTP utilise une base fichier pour démontrer la persistance après redémarrage.


# Exercices

Faire une première tentative sans consulter les corrigés.

## Exercice 1 — Calculer le budget

Avec max_attempts=3, indiquer les dates des trois appels si le premier a lieu à t=1000 et tous échouent.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 2 — Classer les erreurs

Classer timeout fournisseur, catégorie inconnue, token invalide et bug TypeError.

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.

## Exercice 3 — Analyser un crash externe

Une publication externe réussit, mais son accusé de réception est perdu. Que change une outbox ?

Livrable : une réponse argumentée et, lorsque pertinent, un extrait de sortie ou un test.


# Préparation à l’entretien

Répondre en deux minutes par question, avec un exemple concret et une limite.

## Question 1

Pourquoi injecter l’horloge ?

## Question 2

Comment éviter une tempête de retries ?


# Challenge

Faire échouer trois fois le générateur, constater la dead letter, remplacer le générateur par le double nominal et effectuer un replay. Vérifier la trace de replay.

## Critères de revue

Reproductibilité, absence d’effet non autorisé, explication des limites et preuves du comportement attendu.


# Références — J4

Sources consultées le 12 octobre 2026. Les pages évoluent; les exports ciblent la version n8n indiquée dans Compose.

- [Documentation n8n](https://docs.n8n.io/) — vocabulaire et référence des nœuds.
- [Webhook](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.webhook/) — réception et modes de réponse.
- [HTTP Request](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.httprequest/) — appels au backend et credentials.
- [Gestion des erreurs](https://docs.n8n.io/flow-logic/error-handling/) — séparation du flux nominal et des erreurs techniques.
- [Wait](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.wait/) — variante d’attente, non implémentée ici.
- [sqlite3 Python](https://docs.python.org/3/library/sqlite3.html) — transactions et connexions.

Lire en priorité les éléments liés à l’objectif du jour; les autres liens permettent d’approfondir. Les choix du moteur pédagogique sont explicités dans le cours et ne constituent pas une spécification de n8n.
